# CEFR 3-Band Classification via a 0-100 Score - 4 Methods

**Business pipeline (fixed):** every method produces a **0-100 score**, then **2 cut-points**
on that score split it into the **3 bands**.

```
features -> model -> class probabilities -> 0-100 score -> 2 cut-points -> 3 bands
```

**Bands:**

| Band | CEFR levels | Code |
|---|---|---|
| 0 | A1, A2 | `A1-A2` |
| 1 | B1 | `B1` |
| 2 | B2, C1, C2 | `B2-C1-C2` |

**Baseline to beat:** 77% (senior's regression models). **Target:** >=82%.

---

## The four methods, and why these four

Two of them are tree ensembles; the other two are deliberately different paradigms, so that
between them they cover the main families you'd want to defend in a review - and so an
ensemble of them is genuinely diverse rather than four versions of the same idea.

| # | Method | Family | Why it's here | Native importance |
|---|---|---|---|---|
| 1 | **Ordinal Random Forest** (Frank-Hall) | Bagged trees | Current best performer; variance reduction is the right medicine at n~220 | Gini / split importance |
| 2 | **Ordinal Boosting** (LightGBM + Frank-Hall) | Boosted trees | Highest accuracy ceiling; the natural counterpart to bagging | Gain importance |
| 3 | **EBM (GA2M)** | Additive glass-box | The only model whose per-section contributions are **exact and native**, not post-hoc | Additive term scores |
| 4 | **LDA (shrinkage)** | Probabilistic linear | Completely unlike trees: generative, very stable at small n, smooth calibrated posteriors | Discriminant coefficients |

**Every method reports its own feature importance in its own section**, plus a roll-up to
assessment sections via `FEATURE_GROUPS`.

**Light hyperparameter tuning** (randomised search inside cross-validation) is applied to each.

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.model_selection import (
    StratifiedKFold, cross_val_predict, RandomizedSearchCV,
)
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    cohen_kappa_score, confusion_matrix, classification_report,
)

import lightgbm as lgb
from interpret.glassbox import ExplainableBoostingClassifier

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

print("lightgbm + interpret loaded | shap:", HAS_SHAP)

## 1. Load your data  <-- FILL THIS IN

Assign your dataframe to `df`. Nothing else in the notebook needs changing here.

In [ ]:
# TODO: initialise your dataframe here
df = None

# e.g. df = pd.read_csv("your_file.csv")
# e.g. df = pd.read_excel("your_file.xlsx")

## 2. Columns  <-- FILL THIS IN

Put your chosen features (one per feature group, 8-11 total) in `FEATURE_COLS`.
Everything else is metadata: `ciid`, `location`, `split`, and the label.

In [ ]:
# TODO: one feature per feature group (8-11 total)
FEATURE_COLS = []

# OPTIONAL: map each feature to its feature group / assessment section, so the
# importance tables in every method can be reported per section.
# Leave empty to treat every feature as its own section.
FEATURE_GROUPS = {}     # e.g. {"lex_div": "Vocabulary", "mlu": "Grammar"}

# ---- metadata columns (edit names if yours differ) ----
ID_COL       = "ciid"
LOCATION_COL = "location"
SPLIT_COL    = "split"
LABEL_COL    = "cefr"       # TODO: your CEFR label column

# values inside SPLIT_COL that define the sets; anything else is dropped (bad flags)
TRAIN_VALUE = "train"
TEST_VALUE  = "test"

META_COLS = [ID_COL, LOCATION_COL, SPLIT_COL, LABEL_COL]

## 3. Configuration

In [ ]:
RANDOM_STATE = 42

# Band definition: A1,A2 -> 0 | B1 -> 1 | B2,C1,C2 -> 2
BAND_MAP = {
    "A1": 0, "A2": 0,
    "B1": 1,
    "B2": 2, "C1": 2, "C2": 2,
}
BAND_NAMES = ["A1-A2", "B1", "B2-C1-C2"]
N_BANDS = 3

# Score anchors: score = sum_k P(band_k) * anchor_k  -> lands in [0, 100].
#
# IMPORTANT: spacing matters, not the absolute values.
#   * EQUALLY spaced anchors (gaps all the same) make the score an affine function of the
#     expected band index. Any two equally spaced sets rank learners identically, so after
#     the cut-points are re-tuned the accuracy is exactly the same - only the printed
#     number changes. Safe to restyle for the business.
#   * UNEQUALLY spaced anchors (e.g. the GSE set below: gaps 21 and 24) weight the bands
#     differently and CAN reorder learners, so accuracy can shift. That is a genuine
#     modelling choice - re-run and compare before adopting it, don't assume it's free.
BAND_ANCHORS = np.array([0.0, 50.0, 100.0])      # equally spaced (gaps 50, 50)
# BAND_ANCHORS = np.array([29.0, 50.0, 74.0])    # CEFR/GSE-flavoured (gaps 21, 24)

OPTIMIZE_METRIC = "accuracy"     # or "balanced_accuracy" if bands are imbalanced

BASELINE_ACC = 0.77              # senior's regression baseline
TARGET_ACC   = 0.82              # goal

# ---- tuning ----
TUNE = True                      # set False to use the hand-picked defaults
TUNE_ITER = 15                   # randomised-search candidates per method
PERM_REPEATS = 20                # permutation-importance repeats

## 4. Build train / test  (shared by all 4 methods)

Rows whose `split` is neither `train` nor `test` (the "bad flags") are dropped.

In [ ]:
assert df is not None, "Assign your dataframe to `df` in section 1."
assert len(FEATURE_COLS) > 0, "Fill FEATURE_COLS in section 2."

missing = [c for c in FEATURE_COLS + META_COLS if c not in df.columns]
assert not missing, f"Columns not found in df: {missing}"


def to_band(s):
    """Map CEFR labels to band 0/1/2. Accepts 6-level strings or already-coded 0/1/2."""
    s = pd.Series(s)
    if s.dtype.kind in "iuf":
        vals = set(pd.unique(s.dropna()))
        if vals <= {0, 1, 2}:
            return s.astype(int).to_numpy()
    key = s.astype(str).str.strip().str.upper().str.replace(" ", "", regex=False)
    mapped = key.map(BAND_MAP)
    bad = key[mapped.isna()].unique()
    assert len(bad) == 0, f"Unmapped label values: {bad}"
    return mapped.astype(int).to_numpy()


split_norm = df[SPLIT_COL].astype(str).str.strip().str.lower()
train_mask = split_norm == TRAIN_VALUE
test_mask  = split_norm == TEST_VALUE
dropped    = (~train_mask & ~test_mask).sum()

train_df = df.loc[train_mask].copy()
test_df  = df.loc[test_mask].copy()

X_train = train_df[FEATURE_COLS].astype(float)
X_test  = test_df[FEATURE_COLS].astype(float)
y_train = to_band(train_df[LABEL_COL])
y_test  = to_band(test_df[LABEL_COL])

print(f"features           : {len(FEATURE_COLS)}  -> {FEATURE_COLS}")
print(f"train / test rows  : {len(X_train)} / {len(X_test)}")
print(f"dropped (bad flags): {dropped}")
print(f"other split values : {sorted(set(split_norm) - {TRAIN_VALUE, TEST_VALUE})}")
print()
print("train band distribution:")
print(pd.Series(y_train).value_counts().sort_index()
        .rename(lambda i: f"{i} = {BAND_NAMES[i]}"))
print()
print("test band distribution:")
print(pd.Series(y_test).value_counts().sort_index()
        .rename(lambda i: f"{i} = {BAND_NAMES[i]}"))

## 5. Core utilities

- `FrankHallOrdinal` - ordinal decomposition: `K-1` binary models for `P(y > k)`, differenced.
- `proba_to_score` - probabilities -> the **0-100 score**.
- `fit_cutpoints` - grid-search the **2 cut-points** on out-of-fold scores.
- `evaluate` - runs the full pipeline for one method and records the result.
- `tune` - light randomised search inside cross-validation.
- `report_importance` - native + permutation importance, rolled up to sections.

In [ ]:
class FrankHallOrdinal(BaseEstimator, ClassifierMixin):
    """Ordinal decomposition (Frank & Hall, 2001).

    For K ordered classes, fit K-1 binary models P(y > k), then difference the
    cumulative probabilities into per-class probabilities.
    """

    def __init__(self, base_estimator=None):
        self.base_estimator = base_estimator

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.K_ = len(self.classes_)
        self.models_ = []
        for k in range(self.K_ - 1):
            yk = (y > self.classes_[k]).astype(int)
            if len(np.unique(yk)) < 2:
                self.models_.append(("const", float(yk[0])))
            else:
                m = clone(self.base_estimator)
                m.fit(X, yk)
                self.models_.append(("model", m))
        return self

    def predict_proba(self, X):
        n = len(X)
        cum = np.zeros((n, self.K_ - 1))
        for k, (kind, m) in enumerate(self.models_):
            cum[:, k] = m if kind == "const" else m.predict_proba(X)[:, 1]
        cum = np.minimum.accumulate(cum, axis=1)   # P(y>k) must be non-increasing

        p = np.zeros((n, self.K_))
        p[:, 0] = 1.0 - cum[:, 0]
        for k in range(1, self.K_ - 1):
            p[:, k] = cum[:, k - 1] - cum[:, k]
        p[:, -1] = cum[:, -1]

        p = np.clip(p, 1e-9, None)
        return p / p.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


def make_pipe(estimator, scale=True):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)


def proba_to_score(proba, anchors=BAND_ANCHORS):
    return np.asarray(proba) @ np.asarray(anchors, dtype=float)


def apply_cutpoints(scores, t1, t2):
    scores = np.asarray(scores)
    return np.where(scores <= t1, 0, np.where(scores <= t2, 1, 2))


def _scorer(name):
    return accuracy_score if name == "accuracy" else balanced_accuracy_score


def fit_cutpoints(scores, y, metric=OPTIMIZE_METRIC, n_grid=120):
    """Exhaustive grid search for the 2 thresholds maximising `metric`."""
    scores = np.asarray(scores)
    sc = _scorer(metric)
    cand = np.unique(np.percentile(scores, np.linspace(0, 100, n_grid)))
    best_val, best_cuts = -1.0, (33.3, 66.7)
    for i in range(len(cand) - 1):
        for j in range(i + 1, len(cand)):
            v = sc(y, apply_cutpoints(scores, cand[i], cand[j]))
            if v > best_val:
                best_val, best_cuts = v, (float(cand[i]), float(cand[j]))
    return best_cuts, best_val


CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
RESULTS, FITTED, IMPORTANCE = [], {}, {}


def tune(pipe, param_dist, n_iter=None, label=""):
    """Light randomised search inside CV. Returns the best pipeline."""
    if not TUNE or not param_dist:
        return pipe
    n_iter = n_iter or TUNE_ITER
    rs = RandomizedSearchCV(pipe, param_dist, n_iter=n_iter, cv=CV,
                            scoring=OPTIMIZE_METRIC, random_state=RANDOM_STATE,
                            n_jobs=-1, refit=True)
    rs.fit(X_train, y_train)
    print(f"  tuned {label}: CV {OPTIMIZE_METRIC} = {rs.best_score_:.3f}")
    for k, v in rs.best_params_.items():
        print(f"    {k.split('__')[-1]:<22} = {v}")
    return rs.best_estimator_


def _metrics(y, pred):
    return dict(
        acc=accuracy_score(y, pred),
        bal_acc=balanced_accuracy_score(y, pred),
        macro_f1=f1_score(y, pred, average="macro"),
        qwk=cohen_kappa_score(y, pred, weights="quadratic"),
        mae=float(np.mean(np.abs(np.asarray(y) - np.asarray(pred)))),
    )


def evaluate(name, model, anchors=BAND_ANCHORS, verbose=True):
    """OOF probabilities -> 0-100 score -> tune 2 cut-points -> apply to test.

    Cut-points come from *out-of-fold* train probabilities. In-sample probabilities
    are optimistically biased, and thresholds fitted to them do not transfer.
    """
    p_oof = cross_val_predict(clone(model), X_train, y_train,
                              cv=CV, method="predict_proba")
    s_oof = proba_to_score(p_oof, anchors)
    (t1, t2), _ = fit_cutpoints(s_oof, y_train)

    model.fit(X_train, y_train)
    p_tr, p_te = model.predict_proba(X_train), model.predict_proba(X_test)
    s_tr, s_te = proba_to_score(p_tr, anchors), proba_to_score(p_te, anchors)

    pred_oof = apply_cutpoints(s_oof, t1, t2)
    pred_tr = apply_cutpoints(s_tr, t1, t2)
    pred_te = apply_cutpoints(s_te, t1, t2)

    y_full = np.concatenate([y_train, y_test])
    pred_full = np.concatenate([pred_tr, pred_te])

    m_te = _metrics(y_test, pred_te)
    argmax_acc = accuracy_score(y_test, np.argmax(p_te, axis=1))

    row = dict(method=name,
               train_acc=accuracy_score(y_train, pred_tr),
               test_acc=m_te["acc"],
               full_acc=accuracy_score(y_full, pred_full),
               oof_acc=accuracy_score(y_train, pred_oof),
               test_bal_acc=m_te["bal_acc"], macro_f1=m_te["macro_f1"],
               qwk=m_te["qwk"], mae=m_te["mae"],
               argmax_acc=argmax_acc, lift_vs_argmax=m_te["acc"] - argmax_acc,
               cut1=t1, cut2=t2)
    RESULTS.append(row)
    FITTED[name] = dict(model=model, score_test=s_te, pred_test=pred_te,
                        score_train=s_tr, pred_train=pred_tr,
                        proba_test=p_te, proba_train=p_tr, proba_oof=p_oof,
                        cuts=(t1, t2))

    if verbose:
        flag = "PASS" if m_te["acc"] >= TARGET_ACC else ("ok" if m_te["acc"] >= BASELINE_ACC else "-")
        print(f"  TRAIN {row['train_acc']:.3f} | TEST {m_te['acc']:.3f} | "
              f"FULL {row['full_acc']:.3f} | OOF {row['oof_acc']:.3f}   [{flag}]")
        print(f"  bal {m_te['bal_acc']:.3f} | macroF1 {m_te['macro_f1']:.3f} | "
              f"QWK {m_te['qwk']:.3f}")
        print(f"  cut-points ({t1:.1f}, {t2:.1f}) | argmax {argmax_acc:.3f} "
              f"| lift {m_te['acc'] - argmax_acc:+.3f}")
    return row


print("utilities ready")

### Importance helpers

Two views are produced for **every** method:

- **Native importance** - whatever the model itself exposes (split gain, additive term
  scores, discriminant coefficients). Cheap, model-specific, and reveals internal structure.
- **Permutation importance on the final band prediction** - shuffle one feature, re-run the
  *entire* pipeline (model -> 0-100 score -> cut-points) and measure the drop in band accuracy.
  Model-agnostic and measures the **actual deliverable**, so it is the one to quote.

Both are rolled up to assessment sections via `FEATURE_GROUPS`.

In [ ]:
def banded_predict(model, X, t1, t2, anchors=BAND_ANCHORS):
    """Full pipeline: probabilities -> 0-100 score -> cut-points -> band."""
    return apply_cutpoints(proba_to_score(model.predict_proba(X), anchors), t1, t2)


def permutation_importance_banded(model, X, y, t1, t2, n_repeats=None, seed=RANDOM_STATE):
    """Permutation importance measured on the FINAL band prediction."""
    n_repeats = n_repeats or PERM_REPEATS
    rng = np.random.default_rng(seed)
    base = accuracy_score(y, banded_predict(model, X, t1, t2))
    recs = []
    for col in X.columns:
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[col] = rng.permutation(Xp[col].to_numpy())
            drops.append(base - accuracy_score(y, banded_predict(model, Xp, t1, t2)))
        recs.append((col, float(np.mean(drops)), float(np.std(drops))))
    return (pd.DataFrame(recs, columns=["feature", "importance", "std"])
              .sort_values("importance", ascending=False).reset_index(drop=True)), base


def frankhall_native_importance(fh, how="gain"):
    """Native importance per cumulative question of a Frank-Hall model."""
    prof = {}
    for k, (kind, m) in enumerate(fh.models_):
        if kind != "model":
            continue
        est = m.steps[-1][1] if hasattr(m, "steps") else m
        vals = None
        if hasattr(est, "booster_"):
            try:
                vals = est.booster_.feature_importance(importance_type=how)
            except Exception:
                vals = None
        if vals is None:
            vals = getattr(est, "feature_importances_", None)
        if vals is not None:
            v = np.asarray(vals, dtype=float)
            prof[f"P(y>{k})"] = v / v.sum() if v.sum() else v
    if not prof:
        return None
    out = pd.DataFrame(prof, index=FEATURE_COLS)
    out["mean"] = out.mean(axis=1)
    return out.sort_values("mean", ascending=False)


def report_importance(name, native=None, native_note=""):
    """Show native + permutation importance for one method, and roll up to sections."""
    fit = FITTED[name]
    t1, t2 = fit["cuts"]
    perm, base = permutation_importance_banded(fit["model"], X_test, y_test, t1, t2)
    perm["section"] = perm["feature"].map(lambda f: FEATURE_GROUPS.get(f, f))
    IMPORTANCE[name] = dict(native=native, permutation=perm)

    if native is not None:
        print(f"NATIVE importance{(' - ' + native_note) if native_note else ''}")
        nat = native.copy()
        nat["section"] = [FEATURE_GROUPS.get(f, f) for f in nat.index]
        display(nat.round(4))

    print(f"\nPERMUTATION importance on the final band prediction "
          f"(baseline test accuracy {base:.3f})")
    print("value = drop in band accuracy when that feature is shuffled")
    display(perm[["feature", "section", "importance", "std"]].round(4))

    weak = perm[perm["importance"] <= 0]["feature"].tolist()
    if weak:
        print(f"contributing nothing here: {weak}")

    if FEATURE_GROUPS:
        sect = (perm.groupby("section")["importance"].agg(["sum", "mean", "count"])
                    .sort_values("sum", ascending=False))
        sect.columns = ["total_importance", "mean_importance", "n_features"]
        print("\nSECTION-level roll-up")
        display(sect.round(4))
    return perm


print("importance helpers ready")

---

# Method 1 - Ordinal Random Forest (Frank-Hall)

*Current best performer.*

### How it works
The three bands are **ordered**, so instead of one 3-class problem we ask **two cumulative
yes/no questions** (Frank-Hall decomposition):

| Model | Question | Positive label |
|---|---|---|
| RF #1 | is this learner **above band 0**? (`y > 0`) | B1, B2-C1-C2 |
| RF #2 | is this learner **above band 1**? (`y > 1`) | B2-C1-C2 |

Each question gets its own Random Forest. A forest grows hundreds of decision trees, each on
a **bootstrap resample** of the rows and each split restricted to a **random subset of
features**. The output probability is simply the fraction of trees voting "yes".

Band probabilities come back by differencing:
`P(band0) = 1 - c0`, `P(band1) = c0 - c1`, `P(band2) = c1`, where `c0 = P(y>0)`, `c1 = P(y>1)`.

### Why it suits this problem
Individual trees overfit badly, but because they are **de-correlated** (different rows,
different features) their errors largely cancel when averaged. Bagging attacks **variance**,
which is the dominant failure mode at n~220. It is also indifferent to your feature groups
being correlated proxies of the same construct - correlation destabilises linear models but
barely troubles a forest.

### Reading its importance
Native importance = how much each feature reduced impurity across all splits, **per
cumulative question**. The two columns often differ, telling you which sections discriminate
at the **low end** (`P(y>0)`) versus the **high end** (`P(y>1)`).

In [ ]:
print("Method 1 - Ordinal Random Forest (Frank-Hall)")

rf = RandomForestClassifier(
    n_estimators=800, max_features="sqrt", min_samples_leaf=3,
    class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1,
)
pipe_rf = make_pipe(FrankHallOrdinal(rf), scale=False)

grid_rf = {
    "model__base_estimator__n_estimators":     [400, 800],
    "model__base_estimator__max_features":     ["sqrt", 0.5, 0.8],
    "model__base_estimator__min_samples_leaf": [1, 2, 3, 5, 8],
    "model__base_estimator__max_depth":        [None, 6, 10],
}
pipe_rf = tune(pipe_rf, grid_rf, label="Ordinal RF")

M1 = "1. Ordinal RF (Frank-Hall)"
evaluate(M1, pipe_rf)

In [ ]:
fh_rf = FITTED[M1]["model"].steps[-1][1]
report_importance(M1,
                  native=frankhall_native_importance(fh_rf),
                  native_note="impurity reduction, per cumulative question")

---

# Method 2 - Ordinal Boosting (LightGBM + Frank-Hall)

### How it works
Same Frank-Hall skeleton as Method 1, but each cumulative question is answered by a
**gradient-boosted ensemble** instead of a forest. Boosting builds trees **sequentially**:
each new tree is fitted to the errors (gradients) the current ensemble still makes, so the
ensemble steadily corrects itself. Trees are kept deliberately shallow (`max_depth` 2-4) -
each is a weak learner, and the strength comes from combining hundreds of them at a low
learning rate.

### Bagging vs boosting - the key contrast to study
- **Random Forest (bagging)** trains trees **in parallel and independently**, then averages.
  It reduces **variance**. Adding more trees never overfits.
- **Boosting** trains trees **sequentially, each depending on the last**. It reduces **bias**.
  Adding more trees *can* overfit, which is why learning rate, depth and `min_child_samples`
  matter so much at n~220.

That difference is exactly why both are here: they fail in different ways, so if they agree
you can trust the answer, and an ensemble of them is genuinely diverse.

### Reading its importance
LightGBM's **gain** importance = the total improvement in the loss brought by all splits on
that feature. It is more informative than a raw split count because it weights each split by
how much it actually helped.

In [ ]:
print("Method 2 - Ordinal Boosting (LightGBM + Frank-Hall)")

gbm = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.03, max_depth=3, num_leaves=7,
    min_child_samples=15, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, reg_lambda=1.0,
    random_state=RANDOM_STATE, verbose=-1,
)
pipe_gbm = make_pipe(FrankHallOrdinal(gbm), scale=False)

grid_gbm = {
    "model__base_estimator__n_estimators":      [200, 400, 600],
    "model__base_estimator__learning_rate":     [0.01, 0.03, 0.05],
    "model__base_estimator__num_leaves":        [3, 7, 15],
    "model__base_estimator__max_depth":         [2, 3, 4],
    "model__base_estimator__min_child_samples": [5, 10, 15, 20],
    "model__base_estimator__reg_lambda":        [0.0, 1.0, 5.0],
}
pipe_gbm = tune(pipe_gbm, grid_gbm, label="Ordinal Boosting")

M2 = "2. Ordinal Boosting (LGBM + Frank-Hall)"
evaluate(M2, pipe_gbm)

In [ ]:
fh_gbm = FITTED[M2]["model"].steps[-1][1]
report_importance(M2,
                  native=frankhall_native_importance(fh_gbm, how="gain"),
                  native_note="LightGBM gain, per cumulative question")

---

# Method 3 - Explainable Boosting Machine (EBM / GA2M)

### How it works
An EBM is a **generalised additive model** built by boosting:

```
score = intercept + f1(x1) + f2(x2) + ... + fp(xp)   [+ a few fi_j(xi, xj) pairs]
```

Each `f_j` is a **shape function** for a single feature, learned by cyclic gradient boosting
on shallow trees: the algorithm boosts on feature 1, then feature 2, then feature 3, round
and round in **round-robin**, with a very low learning rate. Because each boosting step only
ever touches one feature, the learned effect of that feature is never entangled with the
others - the model stays exactly additive. Optional pairwise terms make it a **GA2M**.

### Why it's here
It is the **only** model in this notebook whose per-section contributions are **exact and
native**. For Random Forest or LightGBM you must *approximate* the explanation afterwards
(SHAP, permutation). For an EBM, `f_j(x_j)` **is** the contribution - there is nothing to
approximate. Given that your business requirement is to explain the score per assessment
section, this is the strongest interpretability story you can bring, and its accuracy is
usually within a point or two of a full black-box booster.

### Reading its importance
Native importance is the **mean absolute contribution** of each term across the data - i.e.
literally how many points, on average, that section moves the score. It is directly
quotable to a stakeholder. Terms containing `&` are pairwise interactions.

In [ ]:
print("Method 3 - Explainable Boosting Machine (GA2M)")

ebm = ExplainableBoostingClassifier(
    interactions=5, outer_bags=8, random_state=RANDOM_STATE,
)
pipe_ebm = make_pipe(ebm, scale=False)

grid_ebm = {
    "model__interactions":     [0, 3, 5, 10],
    "model__learning_rate":    [0.01, 0.02, 0.05],
    "model__max_bins":         [128, 256],
    "model__min_samples_leaf": [2, 5, 10],
}
pipe_ebm = tune(pipe_ebm, grid_ebm, n_iter=min(TUNE_ITER, 8), label="EBM")

M3 = "3. EBM (GA2M)"
evaluate(M3, pipe_ebm)

In [ ]:
ebm_fitted = FITTED[M3]["model"].steps[-1][1]
g = ebm_fitted.explain_global().data()
terms = pd.DataFrame({"term": g["names"], "importance": g["scores"]})
terms["kind"] = np.where(terms["term"].str.contains("&"), "interaction", "main effect")
terms = terms.sort_values("importance", ascending=False).reset_index(drop=True)

print("EBM native term importances (mean absolute contribution to the score)")
display(terms.round(4))

main = (terms[terms["kind"] == "main effect"]
        .set_index("term")[["importance"]])
main = main.reindex(FEATURE_COLS).fillna(0.0)
report_importance(M3, native=main, native_note="EBM additive term score (exact)")

---

# Method 4 - Linear Discriminant Analysis (shrinkage)

### How it works
LDA is a **generative** classifier - the only one here that models how the data was
*produced* rather than drawing a boundary directly. It assumes each band's features follow a
multivariate normal distribution, and that **all three bands share the same covariance
matrix**. Given that, Bayes' rule turns the class densities into posterior probabilities, and
the shared covariance makes the resulting decision boundaries **linear**.

**Shrinkage** (`shrinkage="auto"`, Ledoit-Wolf) pulls the estimated covariance matrix toward a
simple diagonal target. With 220 rows and 8-11 correlated features the raw covariance estimate
is noisy and nearly singular; shrinkage stabilises it automatically, with no tuning needed.

### Why it's here
It is the **opposite** of the tree methods in every way that matters: linear not piecewise
constant, generative not discriminative, a handful of parameters not thousands. That makes it
- an honest check on whether the tree models' extra complexity is buying anything real,
- extremely stable at small n (very low variance),
- a producer of **smooth, well-calibrated posteriors**, which makes for a particularly
  well-behaved 0-100 score - no clumping, no hard 0/1 confidences.

If LDA matches the forests, your problem is close to linearly separable and the simplest model
should probably win on defensibility.

### Reading its importance
Native importance is the **magnitude of the discriminant coefficients**. Because features are
standardised first, the coefficients are directly comparable - a larger absolute value means
that section moves the discriminant more per standard deviation. Signs show direction.

In [ ]:
print("Method 4 - LDA (shrinkage)")

lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
pipe_lda = make_pipe(lda, scale=True)

grid_lda = {"model__shrinkage": ["auto", 0.05, 0.1, 0.2, 0.4, 0.6, 0.8]}
pipe_lda = tune(pipe_lda, grid_lda, n_iter=min(TUNE_ITER, 7), label="LDA")

M4 = "4. LDA (shrinkage)"
evaluate(M4, pipe_lda)

In [ ]:
lda_fitted = FITTED[M4]["model"].steps[-1][1]
coefs = pd.DataFrame(lda_fitted.coef_.T, index=FEATURE_COLS,
                     columns=[f"coef[{b}]" for b in BAND_NAMES])
coefs["mean_abs"] = np.abs(lda_fitted.coef_).mean(axis=0)
coefs = coefs.sort_values("mean_abs", ascending=False)

report_importance(M4, native=coefs,
                  native_note="discriminant coefficients (standardised features)")

---

# Results

## Leaderboard

| Column | Meaning | Honest generalisation estimate? |
|---|---|---|
| `train_acc` | in-sample: fitted on train, predicted on train | **No** - optimistic by construction |
| `test_acc`  | held-out test set | **Yes** |
| `full_acc`  | train + test combined (whole dataset) | **No** - ~2/3 of it is the in-sample train part |
| `oof_acc`   | out-of-fold predictions on train | **Yes** - the most stable one |

- `train_acc` and `full_acc` are **fit diagnostics, not performance claims.**
- A large `train_acc - oof_acc` gap = overfitting.
- **Pick on `oof_acc`, report `test_acc`.** Choosing by highest `test_acc` overfits the test set.

In [ ]:
lb = (pd.DataFrame(RESULTS).sort_values("test_acc", ascending=False)
        .reset_index(drop=True))
lb["vs_baseline"] = lb["test_acc"] - BASELINE_ACC
lb["status"] = np.where(lb["test_acc"] >= TARGET_ACC, "PASS >=82%",
                 np.where(lb["test_acc"] >= BASELINE_ACC, "beats 77%", "below 77%"))

pd.set_option("display.width", 200, "display.max_columns", 50)
display(lb[["method", "train_acc", "test_acc", "full_acc", "oof_acc",
            "test_bal_acc", "macro_f1", "qwk", "mae",
            "argmax_acc", "lift_vs_argmax", "cut1", "cut2",
            "vs_baseline", "status"]].round(4))

print(f"\nbaseline {BASELINE_ACC:.0%} | target {TARGET_ACC:.0%}")
print(f"best by test: {lb.iloc[0]['method']} -> {lb.iloc[0]['test_acc']:.3f}")
best_oof = lb.sort_values("oof_acc", ascending=False).iloc[0]
print(f"best by OOF : {best_oof['method']} -> {best_oof['oof_acc']:.3f}  <- use this to choose")

print("\noverfitting gap (train_acc - oof_acc):")
display((lb[["method", "train_acc", "oof_acc"]]
           .assign(overfit_gap=lb["train_acc"] - lb["oof_acc"])
           .sort_values("overfit_gap", ascending=False)).round(4))

## Does the 0-100 score beat plain `argmax`?

Positive `lift_vs_argmax` means the score + tuned cut-points genuinely add classification
power over reading the model's raw prediction - i.e. the business requirement is also
earning its keep technically.

In [ ]:
crux = lb[["method", "test_acc", "argmax_acc", "lift_vs_argmax"]].sort_values(
    "lift_vs_argmax", ascending=False)
display(crux.round(4))
print(f"mean lift: {crux['lift_vs_argmax'].mean():+.4f}")
print(f"methods helped by the score: {(crux['lift_vs_argmax'] > 0).sum()} / {len(crux)}")

## Feature importance across all 4 methods

In [ ]:
comp = pd.DataFrame(index=FEATURE_COLS)
for name, d in IMPORTANCE.items():
    comp[name] = d["permutation"].set_index("feature")["importance"]
comp["mean"] = comp.mean(axis=1)
comp["agreement"] = (comp[list(IMPORTANCE)] > 0).sum(axis=1)
comp["section"] = [FEATURE_GROUPS.get(f, f) for f in comp.index]
comp = comp.sort_values("mean", ascending=False)

print("Permutation importance (drop in band accuracy) - all methods side by side")
print("'agreement' = how many of the 4 methods found the feature useful\n")
display(comp.round(4))

always = comp[comp["agreement"] == len(IMPORTANCE)].index.tolist()
never = comp[comp["agreement"] == 0].index.tolist()
print(f"useful in ALL methods  : {always}")
print(f"useful in NO method    : {never}   <- drop candidates")

if FEATURE_GROUPS:
    print("\nSection-level, averaged across methods:")
    display(comp.groupby("section")["mean"].agg(["sum", "mean", "count"])
                .sort_values("sum", ascending=False).round(4))

## Confusion matrix for the best method

In [ ]:
best_name = lb.iloc[0]["method"]
best = FITTED[best_name]
print(f"=== {best_name} ===  cut-points: {best['cuts'][0]:.1f}, {best['cuts'][1]:.1f}\n")
display(pd.DataFrame(confusion_matrix(y_test, best["pred_test"]),
                     index=[f"true {b}" for b in BAND_NAMES],
                     columns=[f"pred {b}" for b in BAND_NAMES]))
print(classification_report(y_test, best["pred_test"], target_names=BAND_NAMES))

## Soft-voting ensemble of all 4

Averaging probabilities from models that fail differently (bagging / boosting / additive /
linear) is the cheapest reliable accuracy gain available.

In [ ]:
members = list(FITTED)
P_oof = np.mean([FITTED[m]["proba_oof"] for m in members], axis=0)
P_tr = np.mean([FITTED[m]["proba_train"] for m in members], axis=0)
P_te = np.mean([FITTED[m]["proba_test"] for m in members], axis=0)

s_oof, s_tr, s_te = proba_to_score(P_oof), proba_to_score(P_tr), proba_to_score(P_te)
(t1e, t2e), _ = fit_cutpoints(s_oof, y_train)

pred_tr, pred_te = apply_cutpoints(s_tr, t1e, t2e), apply_cutpoints(s_te, t1e, t2e)
m = _metrics(y_test, pred_te)

print(f"ENSEMBLE of {len(members)}: {members}")
print(f"  TRAIN {accuracy_score(y_train, pred_tr):.3f} | TEST {m['acc']:.3f} | "
      f"OOF {accuracy_score(y_train, apply_cutpoints(s_oof, t1e, t2e)):.3f}")
print(f"  bal {m['bal_acc']:.3f} | macroF1 {m['macro_f1']:.3f} | QWK {m['qwk']:.3f}")
print(f"  cut-points ({t1e:.1f}, {t2e:.1f})")
print(f"\nbest single method: {lb.iloc[0]['test_acc']:.3f} "
      f"-> ensemble: {m['acc']:.3f}  ({m['acc'] - lb.iloc[0]['test_acc']:+.3f})")

## Final 0-100 scores for the test set

The business deliverable: a per-row 0-100 score plus the band it falls into.

In [ ]:
out = pd.DataFrame({
    ID_COL:       test_df[ID_COL].values,
    LOCATION_COL: test_df[LOCATION_COL].values,
    "score_0_100": np.round(FITTED[best_name]["score_test"], 2),
    "pred_band":   [BAND_NAMES[i] for i in FITTED[best_name]["pred_test"]],
    "true_band":   [BAND_NAMES[i] for i in y_test],
})
out["correct"] = out["pred_band"] == out["true_band"]
display(out.head(20))

print(f"\nmethod: {best_name}")
print(f"cut-points: {FITTED[best_name]['cuts'][0]:.1f} / {FITTED[best_name]['cuts'][1]:.1f}")
print(f"accuracy: {out['correct'].mean():.3f}")

# out.to_csv("test_scores.csv", index=False)

---

# Appendix - the 0-100 conversion and the split points

## A.1 The 0-100 conversion

The score is the **expected value of the band index**, rescaled to 0-100:

```
score = P(band 0) x 0  +  P(band 1) x 50  +  P(band 2) x 100
```

For the two Frank-Hall methods this collapses to something much easier to explain:

```
score = 50 x ( P(y > 0) + P(y > 1) )
```

In words: **each of the two thresholds a learner clears is worth 50 points, weighted by how
confident the model is that they cleared it.** Certain to have passed both -> 100; certain to
have passed neither -> 0; genuinely borderline on the first only -> ~25.

**Why an expected value rather than `argmax`.** `argmax` discards confidence: two learners
both labelled "B1" look identical, though one may sit a hair below B2-C1-C2 and the other a
hair above A1-A2. The expected value separates them - which is exactly what a 0-100 business
score is for, and what lets tuned cut-points recover accuracy `argmax` leaves behind.

**Properties.** Bounded in [0,100] (a convex combination of the anchors); monotone (shifting
probability mass upward can only raise it); smooth (small vote changes -> small score changes).

**Anchors: spacing matters, absolute values do not.**
With **equally spaced** anchors the score is `(gap) x E[band index] + offset` - an affine
function of the expected band index. So `[0,50,100]`, `[0,1,2]` and `[10,55,100]` all rank
learners **identically**, and once the cut-points are re-tuned the accuracy is **exactly the
same**; only the printed number changes. Restyling an equally spaced scale for the business
is free.

**Unequally spaced anchors are NOT free.** The GSE-flavoured set `[29,50,74]` has gaps of 21
and 24, so it weights band 2 relatively more than band 1 and **can reorder learners**:

| learner | P(A1-A2) | P(B1) | P(B2-C1-C2) | score `[0,50,100]` | score `[29,50,74]` |
|---|---|---|---|---|---|
| A | 0.601 | 0.00 | 0.399 | **39.90** | **46.96** |
| B | 0.200 | 0.80 | 0.000 | **40.00** | **45.80** |

Under equal spacing A < B; under GSE A > B. The ranking flips, so the band assignment - and
therefore the accuracy - **can differ**. Choosing unequal anchors is a genuine modelling
decision: re-run and compare, do not assume it is cost-free.

## A.2 How the two split points are decided

**Why not fixed 33.3 / 66.7?** The score distribution is not uniform and model confidence is
not calibrated to the band prior. Scores clump; a fixed threshold can slice straight through
the densest region. The thresholds must sit where the model **actually** separates the bands.

**The search.**
1. Take 120 evenly spaced **percentiles** of the score distribution as candidates -
   percentiles concentrate resolution where the data actually is, unlike a flat grid.
2. Enumerate every ordered pair `(t1 < t2)` - roughly 7,000 combinations.
3. Score each by the accuracy of `band = 0 if s<=t1, 1 if s<=t2, else 2`.
4. Keep the best. Exhaustive over the grid, so no local-optimum risk.

`OPTIMIZE_METRIC` controls step 3 - use `balanced_accuracy` if the bands are imbalanced,
otherwise thresholds drift to favour the largest band.

**Fitted on OUT-OF-FOLD scores - the critical detail.** An in-sample score comes from a row
the model was *trained on*, so it is unrealistically confident; thresholds tuned against
those sit in the wrong place for unseen data. The cut-points here are therefore fitted to
5-fold out-of-fold predictions. The `train_acc - oof_acc` gap in the leaderboard shows how
large that in-sample optimism is for each model.

**No leakage.** The chosen `(t1, t2)` are frozen and applied to test unchanged. In production
they are just two constants.

## A.3 Is the optimum a plateau or a spike?
A single best pair means little if accuracy collapses when a threshold shifts by a point.
The next cell measures how wide the near-optimal region is: a **broad plateau** means the
thresholds are stable and will transfer; a **narrow spike** means they are fitted to noise.

In [ ]:
sel = best_name
s_oof_best = proba_to_score(FITTED[sel]["proba_oof"])
t1b, t2b = FITTED[sel]["cuts"]
cand = np.unique(np.percentile(s_oof_best, np.linspace(0, 100, 120)))

rows = []
for a in range(len(cand) - 1):
    for b in range(a + 1, len(cand)):
        rows.append((cand[a], cand[b],
                     accuracy_score(y_train, apply_cutpoints(s_oof_best, cand[a], cand[b]))))
grid = (pd.DataFrame(rows, columns=["t1", "t2", "oof_acc"])
          .sort_values("oof_acc", ascending=False).reset_index(drop=True))

print(f"{sel}: searched {len(grid):,} threshold pairs\n")
display(grid.head(10).round(3))

best_v = grid["oof_acc"].iloc[0]
w1 = (grid["oof_acc"] >= best_v - 0.01).sum()
print(f"best OOF accuracy    : {best_v:.3f}")
print(f"pairs within 1 point : {w1:,}  ({w1/len(grid):.1%} of the grid)")
print(f"chosen cut-points    : {t1b:.2f} / {t2b:.2f}")

top = grid[grid["oof_acc"] >= best_v - 0.01]
print(f"\nplateau spans t1 in [{top['t1'].min():.1f}, {top['t1'].max():.1f}], "
      f"t2 in [{top['t2'].min():.1f}, {top['t2'].max():.1f}]")
print("wide plateau => stable thresholds; narrow spike => fitted to noise.")

## A.4 The one-paragraph version (for the senior)

> Each learner's section features are fed to a model that estimates the probability of being
> above each of the two band boundaries. Those probabilities are combined into a single
> **0-100 score** - effectively, each proficiency threshold the learner clears is worth 50
> points, weighted by the model's confidence that they cleared it. The score is then split
> into the three bands by two cut-points, chosen by exhaustively testing every candidate pair
> against held-out cross-validation predictions on the training data and keeping the pair that
> classified the most learners correctly. Those cut-points are then fixed constants applied
> unchanged to new learners.

**Three things to be ready to defend:**
1. **Why the 0-100 step helps** - the `lift_vs_argmax` column. Positive means it genuinely
   beats reading the model's raw prediction; ~0 means it is the required business output but
   is not adding accuracy. Say which honestly.
2. **Why the thresholds are not round numbers** - they sit where the model actually separates
   the bands; A.3 shows how far they could move without hurting.
3. **Which accuracy is real** - `test_acc` / `oof_acc`. Never `full_acc`, which is inflated by
   in-sample training rows.